In [1]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

DATA = "../data/"

countries  = pd.read_csv(DATA + "clean_countries.csv")
indicators = pd.read_csv(DATA + "clean_indicators.csv")
debt_data  = pd.read_csv(DATA + "clean_debt_data.csv")

print(countries.shape, indicators.shape, debt_data.shape)

(120, 4) (576, 4) (1060074, 4)


In [2]:
full = (debt_data
        .merge(countries, on="country_code")
        .merge(indicators, on="indicator_code"))

print(full.shape)
full.head()

(1060074, 10)


,country_code,indicator_code,year,value,country_name,region,income_group,indicator_name,topic,is_usd
0,ALB,DT.GPA.DPPG,2000,10.5454,Republic of Albania,Europe & Central Asia,Upper middle income,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms,0
1,ALB,DT.GPA.OFFT,2000,10.5454,Republic of Albania,Europe & Central Asia,Upper middle income,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms,0
2,ALB,DT.GPA.PRVT,2000,0.0000,Republic of Albania,Europe & Central Asia,Upper middle income,Average grace period on new external debt comm...,Economic Policy & Debt: External debt: Terms,0
3,ALB,DT.GRE.DPPG,2000,25.7609,Republic of Albania,Europe & Central Asia,Upper middle income,Average grant element on new external debt com...,Economic Policy & Debt: External debt: Terms,0
4,ALB,DT.GRE.OFFT,2000,25.7609,Republic of Albania,Europe & Central Asia,Upper middle income,Average grant element on new external debt com...,Economic Policy & Debt: External debt: Terms,0


In [3]:
full.columns.tolist()



['country_code',
 'indicator_code',
 'year',
 'value',
 'country_name',
 'region',
 'income_group',
 'indicator_name',
 'topic',
 'is_usd']

In [4]:
DEBT_CODE = "DT.DOD.DECT.CD"   # total external debt stock, current US$
YEAR = 2023

debt_year = full[(full["indicator_code"] == DEBT_CODE) & (full["year"] == YEAR)].copy()
debt_year["debt_bn"] = debt_year["value"] / 1e9     # billions are easier to read

print("Countries:", len(debt_year))
print("Total debt (trillion USD):", round(debt_year["value"].sum() / 1e12, 2))

Countries: 120
Total debt (trillion USD): 8.84


In [5]:
debt_year["debt_bn"].describe().round(1)

count     120.0
mean       73.7
std       247.7
min         0.2
25%         3.4
50%        12.5
75%        43.3
max      2450.0
Name: debt_bn, dtype: float64

In [7]:
top10 = debt_year.nlargest(10, "debt_bn")[["country_name", "region", "debt_bn"]]
print(top10)

fig = px.bar(top10.sort_values("debt_bn"), x="debt_bn", y="country_name",
             orientation="h", title=f"Top 10 countries by external debt, {YEAR} (billion USD)")
fig.show()

                          country_name                     region  debt_bn
981909      People's Republic of China        East Asia & Pacific   2450.0
992062               Republic of India                 South Asia    648.0
979243   Federative Republic of Brazil  Latin America & Caribbean    607.0
999200           United Mexican States  Latin America & Caribbean    596.0
1013283            Republic of Türkiye      Europe & Central Asia    500.0
992470           Republic of Indonesia        East Asia & Pacific    400.0
975191              Argentine Republic  Latin America & Caribbean    253.0
982341            Republic of Colombia  Latin America & Caribbean    196.0
1011648            Kingdom of Thailand        East Asia & Pacific    195.0
1014421                        Ukraine      Europe & Central Asia    179.0


In [9]:
bottom10 = debt_year.nsmallest(10, "debt_bn")[["country_name", "region", "debt_bn"]]
print(bottom10)
fig = px.bar(bottom10.sort_values("debt_bn"), x="debt_bn", y="country_name",
             orientation="h", title=f"Bottom 10 countries by external debt, {YEAR} (billion USD)")
fig.show()

                                         country_name  \
1012583                              Kingdom of Tonga   
1011984            Democratic Republic of Timor-Leste   
1006592  Democratic Republic of São Tomé and Principe   
982741                           Union of the Comoros   
1006296                    Independent State of Samoa   
1015215                           Republic of Vanuatu   
1007968                               Solomon Islands   
984435                       Commonwealth of Dominica   
1009653                St. Vincent and the Grenadines   
986750                               State of Eritrea   

                            region   debt_bn  
1012583        East Asia & Pacific  0.196453  
1011984        East Asia & Pacific  0.305742  
1006592         Sub-Saharan Africa  0.318481  
982741          Sub-Saharan Africa  0.381730  
1006296        East Asia & Pacific  0.432495  
1015215        East Asia & Pacific  0.470664  
1007968        East Asia & Pacific  0.52781

In [10]:
top5_share = top10["debt_bn"].head(5).sum() / debt_year["debt_bn"].sum()
print(f"Top 5 countries hold {top5_share:.1%} of the total")

Top 5 countries hold 54.3% of the total


In [11]:
by_region = (debt_year.groupby("region")["debt_bn"].sum()
             .sort_values(ascending=False).reset_index())
print(by_region.round(0))

px.bar(by_region, x="region", y="debt_bn",
       title=f"External debt by region, {YEAR} (billion USD)").show()

                       region  debt_bn
0         East Asia & Pacific   3420.0
1   Latin America & Caribbean   2010.0
2       Europe & Central Asia   1136.0
3          Sub-Saharan Africa    871.0
4                  South Asia    827.0
5  Middle East & North Africa    578.0


In [12]:
by_income = (debt_year.groupby("income_group")["debt_bn"]
             .agg(total="sum", countries="count").round(0))
print(by_income)

                      total  countries
income_group                          
High income             3.0          1
Low income            202.0         23
Lower middle income  2262.0         46
Unclassified           33.0          1
Upper middle income  6341.0         49


In [13]:
trend = (debt_data[debt_data["indicator_code"] == DEBT_CODE]
         .groupby("year")["value"].sum() / 1e12).reset_index()

px.line(trend, x="year", y="value",
        title="Total external debt of 120 countries (trillion USD)").show()
trend.iloc[[0, 10, 20, 23]]

,year,value
0,2000,1.832122
10,2010,3.837514
20,2020,8.152221
23,2023,8.841006


In [15]:
size_band = pd.cut(
    debt_year["debt_bn"],
    bins=[0, 1, 10, 100, 1000, 10000],
    labels=["< 1 bn", "1-10 bn", "10-100 bn", "100 bn - 1 tn", "> 1 tn"],
    right=False,
)

band_counts = size_band.value_counts().sort_index().reset_index()
band_counts.columns = ["size_band", "countries"]
print(band_counts)

px.bar(band_counts, x="size_band", y="countries",
       title=f"How many countries in each debt size band, {YEAR}").show()

       size_band  countries
0         < 1 bn         11
1        1-10 bn         41
2      10-100 bn         50
3  100 bn - 1 tn         17
4         > 1 tn          1


In [16]:
print(indicators["is_usd"].value_counts())
print(indicators["topic"].value_counts().head(5))

is_usd
1    538
0     38
Name: count, dtype: int64
topic
Unknown                                                    73
Economic Policy & Debt: External debt: Debt outstanding    67
Economic Policy & Debt: External debt: Net flows           64
Economic Policy & Debt: External debt: Interest            61
Economic Policy & Debt: External debt: Debt service        58
Name: count, dtype: int64
